# In this notebook, we'll use the modeling module to train and register models

In [1]:
import warnings

warnings.filterwarnings("ignore")

## Testing the Data Modeling Pipeline

In [2]:
import os
import sys

project_root_directory = os.getcwd().rsplit("notebooks", 1)[0]
sys.path.insert(0, project_root_directory)

In [3]:
from core.modeling.config import ModelingConfig, MethodConfig
from core.modeling.pipeline import ModelingPipelineBuilder
import pandas as pd

In [5]:
# Create the MLPipelineConfig object
config = ModelingConfig(
    model_name="MultiClassClassifier",
    data_preprocessing_steps=[
        # MethodConfig(name="sklearn.preprocessing.Normalizer", params=dict(norm="l2")),
    ],
    model_estimator=MethodConfig(
        name="lightgbm.LGBMClassifier",
        params=dict(
            objective="multiclass", num_class=5, n_estimators=100, random_state=42
        ),
    ),
    track_experiment=False,
)

In [38]:
config

ModelingConfig(model_name='MultiClassClassifier', evaluation_metric='', model_estimator=MethodConfig(name='lightgbm.LGBMClassifier', params={'objective': 'multiclass', 'num_class': 5, 'n_estimators': 100, 'random_state': 42}), data_preprocessing_steps=[], track_experiment=False, experiment_name='experiment-mobile-price', run_name='mobile-price', task='regression')

### Simple Usage 

In [6]:
from dotenv import load_dotenv

load_dotenv()
subscription_id = os.environ.get("subscription")
resource_group = os.environ.get("resource_group")
workspace = os.environ.get("workspace")
client_id = os.environ.get("client_id")

In [7]:
from azure.ai.ml import MLClient
from azure.identity import ManagedIdentityCredential

credential = ManagedIdentityCredential(client_id=None)

ml_client = MLClient(credential, subscription_id, resource_group, workspace)

INFO:azure.identity._credentials.managed_identity:ManagedIdentityCredential will use Azure ML managed identity


In [8]:
ws = ml_client.workspaces.get(workspace)
print(ws.location, ":", ws.resource_group)

INFO:azure.identity._internal.msal_managed_identity_client:AzureMLCredential.get_token_info succeeded
INFO:azure.identity._internal.decorators:ManagedIdentityCredential.get_token_info succeeded


eastus : rg-venv


In [ ]:
import mltable

registered_data_asset = ml_client.data.get(name="data", version=2)
tbl = mltable.load(f"azureml:/{registered_data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

In [14]:
pipeline_builder = ModelingPipelineBuilder(config)

In [15]:
pipeline_builder.pipeline

Pipeline(steps=[('LGBMClassifier',
                 LGBMClassifier(num_class=5, objective='multiclass',
                                random_state=42))])

In [16]:
X = df.drop(columns=["anomalyscore"])
y = df["anomalyscore"]

In [17]:
X_train, X_test, y_train, y_test = pipeline_builder.split_data_stratified(X, y)

In [ ]:
pipeline_builder.X_train

In [ ]:
pipeline_builder.fit()

test_score = pipeline_builder.pipeline.score(X_test, y_test)
train_score = pipeline_builder.pipeline.score(X_train, y_train)
print(f"Test Score: {test_score}")
print(f"Train Score: {train_score}")




## Checking for existing de env

Let's explore the environments within the workspace.


> **Note**:
> If the **azure-ai-ml** package is not installed, run `pip install azure-ai-ml` to install it.

In [ ]:
envs = ml_client.environments.list()
for env in envs:
    print(env.name)

Submitting the job with the new custom environment triggers the build of the environment. The first time you use a newly created environment, it can take 10-15 minutes to build the environment, which also means your job will take longer to complete.
You can also choose to manually trigger the build of the environment before you submit a job. The environment only needs to be built the first time you use it.

## Creating a job to use a data asset

After using a notebook for experimentation. You can use scripts to train machine learning models. A script can be run as a job, and for each job you can specify inputs and outputs. 

You can use either **data assets** or **datastore paths** as inputs or outputs of a job. Also, it is possible to read these data directly from the job.

The cells below creates the **main.py** script in the **src** folder. 

In [ ]:
%%writefile ../main.py


import sys
import os

# Add the parent directory to sys.path
current_path = os.path.dirname(os.path.abspath(__file__))
import mltable


from core.modeling.config import ModelingConfig, MethodConfig
from core.modeling.pipeline import ModelingPipelineBuilder
import pandas as pd

# Create the MLPipelineConfig object
config = ModelingConfig(
    model_name="Linear Regression",
    data_preprocessing_steps=[
        MethodConfig(
            name="sklearn.preprocessing.Normalizer", params=dict(norm="l2")
        ),
    ],
    model_estimator=MethodConfig(
        name="sklearn.linear_model.Ridge", params=dict(alpha=0.9)
    ),
    track_experiment=True
)


tbl = mltable.load(f"azureml://...path...")
df = tbl.to_pandas_dataframe()
X = df.drop(["Price"], axis = 1)
y = pd.DataFrame(df["Price"].copy())

pipeline_builder = ModelingPipelineBuilder(config)
pipeline_builder.split_data(X, y)

pipeline_builder.fit()


To submit a job that runs the **main.py** script, run the cell below. 

The job is configured to use the data asset `diabetes-local`, pointing to the local **mobile-price-local.csv** file as input. The output is a path pointing to a folder in the new datastore `blob_mobileprice_cleaned`.

In [ ]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import command

# configure job
job = command(
    code="../",
    command="python main.py",
    environment="docker-context-repo-based-v1:1",
    compute="sandbox-ci",
    display_name="training-mobile-data",  # if we dont define it, it will be the run name definition
    experiment_name="mobile-price-exp",
)

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor the job at", aml_url)

## Creating a component to execute a pipeline  (using yaml definition)

In [ ]:
%%writefile example_configs/config.yml
# <component>
$schema: https://azuremlschemas.azureedge.net/latest/commandJob.schema.json
name: train_credit_defaults_model
display_name: training-mobile-data
description: Training job for mobile price data
# version: 1 # Not specifying a version will automatically update the version
code: .
environment: azureml:docker-context-repo-based-v1:1
command: >-
  python main.py 
# </component>

Optionally, register the component in the workspace for future reuse.

In [ ]:
# importing the Component Package
from azure.ai.ml import load_component

# Loading the component from the yml file
train_component = load_component(source=os.path.join("../", "config.yml"))

In [ ]:
# Create (register) the component in your workspace
print(
    f"Component {train_component.name} and {train_component.command} with Version {train_component.version} is registered"
)

In [ ]:
# the dsl decorator tells the sdk that we are defining an Azure ML pipeline
from azure.ai.ml import dsl, Input, Output


@dsl.pipeline(
    compute="sandbox-demo-ci",
    description="training pipeline",
)
def mobile_defaults_pipeline():

    # using train_func like a python call with its own inputs
    train_job = train_component()

    # a pipeline returns a dictionary of outputs
    # keys will code for the pipeline output identifier

In [ ]:
# Let's instantiate the pipeline with the parameters of our choice
pipeline = mobile_defaults_pipeline()

In [ ]:
import webbrowser

# submit the pipeline job
pipeline_job = ml_client.jobs.create_or_update(
    pipeline,
    experiment_name="mobile-price-exp",
    # Project's name
)
# open the pipeline in web browser
webbrowser.open(pipeline_job.studio_url)